# TASK 4 — Merge Indicator Results with Probe Performance

**Goal**: Prepare a unified table for plotting probe vs indicator evidence.

## Steps
1. Load linear probe R² and MLP probe R² results
2. Merge with NN indicator table by descriptor name
3. Compute:
   - MLP gain = R²_MLP − R²_linear
4. Flag descriptors:
   - **high-linear**: R²_linear > threshold
   - **non-linear**: MLP gain > threshold
   - **low-probe but strong-indicator**: ratio_k50 > threshold & R²_linear < threshold

## Outputs
- CSV: `results/indicators/probe_indicator_merged.csv`
- Summary table with descriptor classifications

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports complete")

✓ Imports complete


## 1. Load Probe Results

In [10]:
# Paths
results_dir = Path('../results')
indicators_dir = results_dir / 'indicators'
indicators_dir.mkdir(parents=True, exist_ok=True)

print(f"Results directory: {results_dir}")
print(f"Indicators directory: {indicators_dir}")

Results directory: ../results
Indicators directory: ../results/indicators


In [11]:
# Load probe results CSV
print("Loading probe results...")

probe_csv_path = results_dir / 'all_descriptors_probing_results.csv'
df_probe = pd.read_csv(probe_csv_path)

print(f"  ✓ Loaded: {probe_csv_path}")
print(f"  Shape: {df_probe.shape}")
print(f"  Columns: {df_probe.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_probe.head())

Loading probe results...
  ✓ Loaded: ../results/all_descriptors_probing_results.csv
  Shape: (201, 9)
  Columns: ['descriptor', 'r2', 'mae', 'n_train', 'n_test', 'train_mean', 'train_std', 'test_mean', 'test_std']

First few rows:
            descriptor        r2        mae  n_train  n_test  train_mean  \
0            LabuteASA  0.545693  30.667478   159271   45185  174.782167   
1                Chi0v  0.536735   3.277886   159271   45185   17.682972   
2  NumValenceElectrons  0.536183  29.589413   159271   45185  161.634824   
3                Chi2n  0.529809   1.669583   159271   45185    8.076316   
4                Chi1n  0.529635   1.931019   159271   45185   10.210974   

   train_std   test_mean   test_std  
0  83.980791  145.061189  64.726362  
1   8.869907   14.504924   6.753155  
2  81.258593  131.729866  62.007571  
3   4.813903    6.513898   3.458788  
4   5.528388    8.226892   3.924748  


In [12]:
# Load detailed probe results (pkl files contain lists of dicts with r2 scores)
print("\nLoading MLP and linear probe results from pkl files...")

try:
    # Load linear probe results
    with open(results_dir / 'all_descriptors_probing_results_linear.pkl', 'rb') as f:
        linear_results = pickle.load(f)
    print(f"  ✓ Loaded linear probe results ({len(linear_results)} descriptors)")
    
    # Load MLP probe results  
    with open(results_dir / 'all_descriptors_probing_results_mlp.pkl', 'rb') as f:
        mlp_results = pickle.load(f)
    print(f"  ✓ Loaded MLP probe results ({len(mlp_results)} descriptors)")
    
    # Convert to dataframes
    df_linear = pd.DataFrame(linear_results)
    df_mlp = pd.DataFrame(mlp_results)
    
    # Rename r2 columns to be specific
    df_linear = df_linear.rename(columns={'r2': 'r2_linear', 'mae': 'mae_linear'})
    df_mlp = df_mlp.rename(columns={'r2': 'r2_mlp', 'mae': 'mae_mlp'})
    
    # Keep only essential columns for merge
    df_linear = df_linear[['descriptor', 'r2_linear', 'mae_linear']]
    df_mlp = df_mlp[['descriptor', 'r2_mlp', 'mae_mlp']]
    
    # Merge linear and MLP results
    df_probes = df_linear.merge(df_mlp, on='descriptor', how='outer')
    
    print(f"\n  Combined probes dataframe shape: {df_probes.shape}")
    print(f"  Columns: {df_probes.columns.tolist()}")
    print(f"\n  Sample rows:")
    print(df_probes.head(3).to_string())
    
    # Merge with main probe dataframe (from CSV)
    df_probe = df_probe.merge(df_probes, on='descriptor', how='left', suffixes=('_csv', ''))
    
    print(f"\n  Merged with CSV data")
    
except FileNotFoundError as e:
    print(f"  ❌ pkl files not found: {e}")
    print(f"  Will use CSV r2 column as linear probe R²")
    df_probe['r2_linear'] = df_probe['r2']
    df_probe['r2_mlp'] = np.nan

print(f"\nFinal probe dataframe shape: {df_probe.shape}")
print(f"Columns: {df_probe.columns.tolist()}")



Loading MLP and linear probe results from pkl files...
  ✓ Loaded linear probe results (201 descriptors)
  ✓ Loaded MLP probe results (201 descriptors)

  Combined probes dataframe shape: (201, 5)
  Columns: ['descriptor', 'r2_linear', 'mae_linear', 'r2_mlp', 'mae_mlp']

  Sample rows:
      descriptor  r2_linear  mae_linear    r2_mlp   mae_mlp
0   BCUT2D_CHGHI   0.315475    0.147402  0.292461  0.147667
1   BCUT2D_CHGLO   0.333331    0.131918  0.305248  0.131477
2  BCUT2D_LOGPHI   0.232772    0.113662  0.262030  0.109645

  Merged with CSV data

Final probe dataframe shape: (201, 13)
Columns: ['descriptor', 'r2', 'mae', 'n_train', 'n_test', 'train_mean', 'train_std', 'test_mean', 'test_std', 'r2_linear', 'mae_linear', 'r2_mlp', 'mae_mlp']


## 2. Load Indicator Results (TASK 2)

In [13]:
# Load indicator results from TASK 2
print("Loading indicator results from TASK 2...")

indicator_csv_path = indicators_dir / 'nn_descriptor_consistency.csv'

try:
    df_indicators = pd.read_csv(indicator_csv_path)
    print(f"  ✓ Loaded: {indicator_csv_path}")
    print(f"  Shape: {df_indicators.shape}")
except FileNotFoundError:
    print(f"  ❌ File not found: {indicator_csv_path}")
    print(f"  Please run TASK 2 notebook first.")
    raise

print(f"\nIndicator columns: {df_indicators.columns.tolist()}")
print(f"Unique k values: {df_indicators['k'].unique()}")
print(f"Unique neighbor types: {df_indicators['neighbors'].unique()}")

Loading indicator results from TASK 2...
  ✓ Loaded: ../results/indicators/nn_descriptor_consistency.csv
  Shape: (1248, 7)

Indicator columns: ['descriptor', 'k', 'neighbors', 'nn_mean_diff', 'random_mean_diff', 'effect_ratio', 'spearman_corr']
Unique k values: [ 10  50 100]
Unique neighbor types: ['inclusive' 'exclusive']


In [14]:
# Filter for k=50, exclusive (primary analysis)
print("\nFiltering indicators for k=50, exclusive...")

df_ind_k50 = df_indicators[
    (df_indicators['k'] == 50) & 
    (df_indicators['neighbors'] == 'exclusive')
].copy()

print(f"  Filtered shape: {df_ind_k50.shape}")
print(f"  Unique descriptors: {df_ind_k50['descriptor'].nunique()}")


Filtering indicators for k=50, exclusive...
  Filtered shape: (208, 7)
  Unique descriptors: 208


## 3. Merge Probe and Indicator Results

In [15]:
# Merge on descriptor name
print("Merging probe and indicator results...")

df_merged = df_probe.merge(
    df_ind_k50[['descriptor', 'effect_ratio', 'spearman_corr', 'nn_mean_diff', 'random_mean_diff']],
    on='descriptor',
    how='left'
)

print(f"  Merged shape: {df_merged.shape}")
print(f"  Descriptors with indicator data: {df_merged['effect_ratio'].notna().sum()}")
print(f"  Descriptors without indicator data: {df_merged['effect_ratio'].isna().sum()}")

if df_merged['effect_ratio'].isna().any():
    print(f"\n  Descriptors missing indicator data:")
    print(df_merged[df_merged['effect_ratio'].isna()]['descriptor'].tolist()[:10])

Merging probe and indicator results...
  Merged shape: (201, 17)
  Descriptors with indicator data: 201
  Descriptors without indicator data: 0


## 4. Compute MLP Gain

In [16]:
# Compute MLP gain
print("Computing MLP gain...")

# Check which columns we have
if 'r2_linear' in df_merged.columns and 'r2_mlp' in df_merged.columns:
    # Compute MLP gain
    df_merged['mlp_gain'] = df_merged['r2_mlp'] - df_merged['r2_linear']
    linear_col = 'r2_linear'
    mlp_col = 'r2_mlp'
    
    print(f"  ✓ Using r2_linear and r2_mlp columns")
    print(f"\n  Linear R² statistics:")
    print(f"    Mean: {df_merged[linear_col].mean():.4f}")
    print(f"    Median: {df_merged[linear_col].median():.4f}")
    print(f"    Range: [{df_merged[linear_col].min():.4f}, {df_merged[linear_col].max():.4f}]")
    
    print(f"\n  MLP R² statistics:")
    print(f"    Mean: {df_merged[mlp_col].mean():.4f}")
    print(f"    Median: {df_merged[mlp_col].median():.4f}")
    print(f"    Range: [{df_merged[mlp_col].min():.4f}, {df_merged[mlp_col].max():.4f}]")
    
    print(f"\n  MLP gain statistics:")
    print(f"    Mean: {df_merged['mlp_gain'].mean():.4f}")
    print(f"    Median: {df_merged['mlp_gain'].median():.4f}")
    print(f"    Range: [{df_merged['mlp_gain'].min():.4f}, {df_merged['mlp_gain'].max():.4f}]")
    print(f"    Positive gains: {(df_merged['mlp_gain'] > 0).sum()} descriptors")
    print(f"    Negative gains: {(df_merged['mlp_gain'] < 0).sum()} descriptors")
    
elif 'r2' in df_merged.columns:
    # Fallback: assume r2 is linear, MLP not available
    print("  ⚠️  Only linear probe R² available (r2 column)")
    df_merged['r2_linear'] = df_merged['r2']
    df_merged['r2_mlp'] = np.nan
    df_merged['mlp_gain'] = np.nan
    linear_col = 'r2_linear'
    mlp_col = 'r2_mlp'
else:
    print("  ❌ Could not find R² columns")
    raise ValueError("Missing R² columns in merged dataframe")


Computing MLP gain...
  ✓ Using r2_linear and r2_mlp columns

  Linear R² statistics:
    Mean: 0.0070
    Median: 0.0547
    Range: [-4.8065, 0.4456]

  MLP R² statistics:
    Mean: 0.1028
    Median: 0.0735
    Range: [-1.9280, 0.5587]

  MLP gain statistics:
    Mean: 0.0959
    Median: 0.0261
    Range: [-0.5758, 4.8533]
    Positive gains: 118 descriptors
    Negative gains: 80 descriptors


## 5. Flag Descriptors

In [17]:
# Define thresholds
print("Flagging descriptors...")

threshold_high_linear = 0.3  # R² > 0.3 = high linear predictability
threshold_mlp_gain = 0.05    # MLP gain > 0.05 = significant non-linearity
threshold_low_probe = 0.1    # R² < 0.1 = low probe performance
threshold_strong_indicator = 1.5  # effect_ratio > 1.5 = strong indicator

print(f"  Thresholds:")
print(f"    High linear: R² > {threshold_high_linear}")
print(f"    Non-linear: MLP gain > {threshold_mlp_gain}")
print(f"    Low probe: R² < {threshold_low_probe}")
print(f"    Strong indicator: effect_ratio > {threshold_strong_indicator}")

# Create flags
df_merged['flag_high_linear'] = df_merged[linear_col] > threshold_high_linear
df_merged['flag_nonlinear'] = (df_merged['mlp_gain'] > threshold_mlp_gain) if 'mlp_gain' in df_merged.columns else False
df_merged['flag_low_probe_strong_indicator'] = (
    (df_merged[linear_col] < threshold_low_probe) & 
    (df_merged['effect_ratio'] > threshold_strong_indicator)
)

print(f"\n  Flag counts:")
print(f"    High linear: {df_merged['flag_high_linear'].sum()}")
print(f"    Non-linear: {df_merged['flag_nonlinear'].sum() if isinstance(df_merged['flag_nonlinear'], pd.Series) else 0}")
print(f"    Low probe but strong indicator: {df_merged['flag_low_probe_strong_indicator'].sum()}")

Flagging descriptors...
  Thresholds:
    High linear: R² > 0.3
    Non-linear: MLP gain > 0.05
    Low probe: R² < 0.1
    Strong indicator: effect_ratio > 1.5

  Flag counts:
    High linear: 33
    Non-linear: 76
    Low probe but strong indicator: 18


## 6. Save Merged Results

In [18]:
# Save to CSV
output_path = indicators_dir / 'probe_indicator_merged.csv'
df_merged.to_csv(output_path, index=False)

print(f"✅ Saved merged results to: {output_path}")
print(f"   Shape: {df_merged.shape}")
print(f"   Columns: {len(df_merged.columns)}")

✅ Saved merged results to: ../results/indicators/probe_indicator_merged.csv
   Shape: (201, 21)
   Columns: 21


## 7. Summary Tables

In [19]:
# Overall summary
print("\n" + "="*80)
print("PROBE vs INDICATOR SUMMARY")
print("="*80)

print(f"\nDataset:")
print(f"  Total descriptors: {len(df_merged)}")
print(f"  With probe data: {df_merged[linear_col].notna().sum()}")
print(f"  With indicator data: {df_merged['effect_ratio'].notna().sum()}")
print(f"  With both: {(df_merged[linear_col].notna() & df_merged['effect_ratio'].notna()).sum()}")

print(f"\nProbe Performance:")
print(f"  Mean R² (linear): {df_merged[linear_col].mean():.4f}")
print(f"  Median R² (linear): {df_merged[linear_col].median():.4f}")
if df_merged['mlp_gain'].notna().any():
    print(f"  Mean MLP gain: {df_merged['mlp_gain'].mean():.4f}")
    print(f"  Median MLP gain: {df_merged['mlp_gain'].median():.4f}")

print(f"\nIndicator Performance:")
print(f"  Mean effect ratio: {df_merged['effect_ratio'].mean():.4f}")
print(f"  Median effect ratio: {df_merged['effect_ratio'].median():.4f}")
print(f"  Mean Spearman: {df_merged['spearman_corr'].mean():.4f}")
print(f"  Median Spearman: {df_merged['spearman_corr'].median():.4f}")

print(f"\nClassifications:")
print(f"  High linear (R² > {threshold_high_linear}): {df_merged['flag_high_linear'].sum()}")
if df_merged['mlp_gain'].notna().any():
    print(f"  Non-linear (gain > {threshold_mlp_gain}): {df_merged['flag_nonlinear'].sum()}")
print(f"  Low probe + strong indicator: {df_merged['flag_low_probe_strong_indicator'].sum()}")

print("\n" + "="*80)


PROBE vs INDICATOR SUMMARY

Dataset:
  Total descriptors: 201
  With probe data: 201
  With indicator data: 201
  With both: 201

Probe Performance:
  Mean R² (linear): 0.0070
  Median R² (linear): 0.0547
  Mean MLP gain: 0.0959
  Median MLP gain: 0.0261

Indicator Performance:
  Mean effect ratio: 1.4412
  Median effect ratio: 1.4187
  Mean Spearman: 0.0443
  Median Spearman: 0.0491

Classifications:
  High linear (R² > 0.3): 33
  Non-linear (gain > 0.05): 76
  Low probe + strong indicator: 18



In [20]:
# Top descriptors by different criteria
print("\n" + "="*80)
print("TOP DESCRIPTORS BY DIFFERENT CRITERIA")
print("="*80)

# Sort by linear R²
df_sorted_r2 = df_merged.sort_values(linear_col, ascending=False)

print(f"\n{'Top 10 by Linear Probe R²':^80}")
print(f"{'Descriptor':<25} {'R² Linear':>10} {'Effect Ratio':>14} {'Spearman':>12}")
print("-" * 80)
for _, row in df_sorted_r2.head(10).iterrows():
    print(f"{row['descriptor']:<25} {row[linear_col]:>10.4f} "
          f"{row['effect_ratio']:>14.4f} {row['spearman_corr']:>12.4f}")

# Sort by effect ratio
df_sorted_ratio = df_merged.sort_values('effect_ratio', ascending=False)

print(f"\n{'Top 10 by Effect Ratio (k=50, exclusive)':^80}")
print(f"{'Descriptor':<25} {'Effect Ratio':>14} {'R² Linear':>10} {'Spearman':>12}")
print("-" * 80)
for _, row in df_sorted_ratio.head(10).iterrows():
    print(f"{row['descriptor']:<25} {row['effect_ratio']:>14.4f} "
          f"{row[linear_col]:>10.4f} {row['spearman_corr']:>12.4f}")

# Low probe but strong indicator
df_anomalies = df_merged[df_merged['flag_low_probe_strong_indicator']].sort_values(
    'effect_ratio', ascending=False
)

if len(df_anomalies) > 0:
    print(f"\n{'Low Probe BUT Strong Indicator (Anomalies)':^80}")
    print(f"{'Descriptor':<25} {'R² Linear':>10} {'Effect Ratio':>14} {'Spearman':>12}")
    print("-" * 80)
    for _, row in df_anomalies.head(10).iterrows():
        print(f"{row['descriptor']:<25} {row[linear_col]:>10.4f} "
              f"{row['effect_ratio']:>14.4f} {row['spearman_corr']:>12.4f}")
else:
    print(f"\nNo descriptors with low probe but strong indicator")

print("\n" + "="*80)


TOP DESCRIPTORS BY DIFFERENT CRITERIA

                           Top 10 by Linear Probe R²                            
Descriptor                 R² Linear   Effect Ratio     Spearman
--------------------------------------------------------------------------------
Chi0                          0.4456         2.1092       0.0930
HeavyAtomMolWt                0.4327         2.0969       0.0982
SMR_VSA1                      0.4215         1.8175       0.0772
Chi1v                         0.4116         2.0485       0.0889
Chi0v                         0.4093         2.0999       0.0914
Chi0n                         0.3966         2.1195       0.0916
LabuteASA                     0.3925         2.1181       0.0959
Chi1                          0.3886         2.1019       0.0930
Chi3v                         0.3880         1.9768       0.0768
NumValenceElectrons           0.3862         2.1277       0.0941

                    Top 10 by Effect Ratio (k=50, exclusive)                    
D

## 8. Correlation Analysis

In [21]:
# Compute correlations between probe and indicator metrics
print("\nCorrelation between probe and indicator metrics:\n")

# Select numeric columns
corr_cols = [linear_col, 'effect_ratio', 'spearman_corr', 'nn_mean_diff']
if df_merged['mlp_gain'].notna().any():
    corr_cols.append('mlp_gain')

df_corr = df_merged[corr_cols].corr()
print(df_corr.round(3))

print(f"\nKey observations:")
r2_ratio_corr = df_merged[[linear_col, 'effect_ratio']].corr().iloc[0, 1]
r2_spearman_corr = df_merged[[linear_col, 'spearman_corr']].corr().iloc[0, 1]

print(f"  Corr(R² linear, effect_ratio): {r2_ratio_corr:.3f}")
print(f"  Corr(R² linear, spearman): {r2_spearman_corr:.3f}")

if r2_ratio_corr > 0.5:
    print(f"  → Strong positive: Probe and indicator agree on descriptor quality")
elif r2_ratio_corr > 0.2:
    print(f"  → Moderate positive: Some agreement between probe and indicator")
else:
    print(f"  → Weak: Probe and indicator measure different aspects")


Correlation between probe and indicator metrics:

               r2_linear  effect_ratio  spearman_corr  nn_mean_diff  mlp_gain
r2_linear          1.000         0.262          0.329         0.005    -0.863
effect_ratio       0.262         1.000          0.743        -0.030     0.048
spearman_corr      0.329         0.743          1.000         0.061    -0.003
nn_mean_diff       0.005        -0.030          0.061         1.000    -0.014
mlp_gain          -0.863         0.048         -0.003        -0.014     1.000

Key observations:
  Corr(R² linear, effect_ratio): 0.262
  Corr(R² linear, spearman): 0.329
  → Moderate positive: Some agreement between probe and indicator


## 9. Task Complete

In [22]:
print("\n" + "="*80)
print("TASK 4 — Probe-Indicator Merge COMPLETE")
print("="*80)
print(f"\nResults saved:")
print(f"  - CSV: results/indicators/probe_indicator_merged.csv")
print(f"\nData ready for:")
print(f"  - Scatter plots: R² vs effect_ratio")
print(f"  - Agreement analysis: Which descriptors have high/low both?")
print(f"  - Anomaly investigation: Low probe but strong indicator")
print(f"  - Non-linearity analysis: MLP gain vs indicator strength")
print("="*80)


TASK 4 — Probe-Indicator Merge COMPLETE

Results saved:
  - CSV: results/indicators/probe_indicator_merged.csv

Data ready for:
  - Scatter plots: R² vs effect_ratio
  - Agreement analysis: Which descriptors have high/low both?
  - Anomaly investigation: Low probe but strong indicator
  - Non-linearity analysis: MLP gain vs indicator strength
